# Edge-deployable auto-encoder with INT8 TFLite export

A compact convolutional auto-encoder (20/40 channels, 32×128 px grayscale input) designed for deployment on a
microcontroller-class device:

* Same reconstruction-based training and Youden-J thresholding as the main auto-encoder notebook.
* The decoder uses **bilinear Upsample + Conv2d instead of ConvTranspose2d**, because transposed convolutions with
  output padding cannot be converted by `ai-edge-torch`.
* The trained PyTorch model is converted with **ai-edge-torch** to a **fully INT8-quantised TFLite model**
  (representative dataset from the training loader, `TFLITE_BUILTINS_INT8`, int8 input/output), plus an optional ONNX export.

**Data:** proprietary production images (anonymised as *Product A*); not included.


In [ ]:
import os

# Root folder of the dataset(s) used in this notebook.
# Set the DATA_ROOT environment variable or edit the default below.
# On Google Colab you can first mount Drive with:
#   from google.colab import drive; drive.mount('/content/drive')
DATA_ROOT = os.environ.get("DATA_ROOT", "./data")
base_path = DATA_ROOT + "/"

%pip install -q torchinfo ai-edge-torch tensorflow onnx

## User-specified parameters
- `dataset_path`: The path to the dataset containing the images to be used.
- `seed`: The seed for random number generation, useful for reproducibility.
- `anomaly_detection`: Set to True or False based on your requirement.
- `subimage_width`: Width for subimages or None.

- `set_params`:
  - `ratio`: The ratio of data to be included in the set.
  - `subimage_shift`: The distance between subimages (irrelevant when not creating subimages).

In [ ]:
import os
import torch
import random
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
import numpy as np
from scipy.ndimage import gaussian_filter
from skimage.exposure import adjust_sigmoid

class PanelsDataset(Dataset):
    """ Custom dataset class for handling image datasets """
    def __init__(self, image_list):
        self.image_list = image_list
        self.transform = transforms.ToTensor()

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        image, label = self.image_list[idx]
        # image = process_and_crop_image(image, method="lower_edge", new_height=130, margin=7, outliers_pixels=10)
        image = self.transform(image)
        return image, label

def create_subimages(image, label, subimage_width, subimage_shift, filename):
    """ Function to create subimages from the given image """
    if subimage_width is None:
        return [(image, label, filename)]

    subimages = []

    '''
    full_width = 800  # Assuming full width of images is 800
    for start in range(0, full_width - subimage_width + 1, subimage_shift):
        subimage = image.crop((start, 0, start + subimage_width, 200))
        subimages.append((subimage, label, filename))
    '''

    image_width, image_height = image.size
    for start in range(0, image_width - subimage_width + 1, subimage_shift):
        subimage = image.crop((start, 0, start + subimage_width, image_height))
        subimages.append((subimage, label, filename))
    return subimages

def create_subimages_and_names(files, filenames, conform_files, subimage_width, subimage_shift):
    """ Create subimages and corresponding names for a set of files """
    images = []
    names = []
    for file, filename in zip(files, filenames):
        subimages = create_subimages(file, 0 if file in conform_files else 1, subimage_width, subimage_shift, filename)
        for img, label, fname in subimages:
            images.append((img, label))
            names.append(fname)
    return images, names

import shutil
from pathlib import Path

def prepare_datasets(
    dataset_path, train_params, valid_params, test_params,
    seed, anomaly_detection, anomaly_detection_couple, subimage_width,
    load_dataset = True):
    torch.manual_seed(seed)  # Set random seed for reproducibility
    random.seed(seed)  # Set random seed for reproducibility

    if load_dataset:
      transform = transforms.ToTensor()  # Common transform for all images
      conform_zip_path = os.path.join(dataset_path, "Zipped", 'normal')
      defect_zip_path = os.path.join(dataset_path, "Zipped", 'defects')

      conform_path = "work/normal"
      defect_path = "work/defects"

      Path(conform_path).mkdir(parents=True, exist_ok=True)
      Path(defect_path).mkdir(parents=True, exist_ok=True)

      print(f"Unzipping {conform_zip_path}")
      shutil.unpack_archive(conform_zip_path + ".zip", "work/normal")
      print(f"Unzipping {defect_zip_path}")
      shutil.unpack_archive(defect_zip_path + ".zip", "work/defects")

      conform_filenames = sorted(os.listdir(conform_path))
      defect_filenames = sorted(os.listdir(defect_path))
      print(conform_filenames)
      conform_files = [Image.open(os.path.join(conform_path, file)).convert('L') for file in conform_filenames]
      defect_files = [Image.open(os.path.join(defect_path, file)).convert('L') for file in defect_filenames]
      for idx in range(len(conform_files)):
        img = conform_files[idx]
        w, h = img.size
        conform_files[idx] = img.crop((0, 0, subimage_width, h))
      for idx in range(len(defect_files)):
        img = defect_files[idx]
        w, h = img.size
        defect_files[idx] = img.crop((0, 0, subimage_width, h))
      print("Load image Ok")

      # Shuffle conform_files and conform_filenames while maintaining correspondence
      conform_data = list(zip(conform_files, conform_filenames))
      random.shuffle(conform_data)
      conform_files[:], conform_filenames[:] = zip(*conform_data)

      # Shuffle defect_files and defect_filenames while maintaining correspondence
      defect_data = list(zip(defect_files, defect_filenames))
      random.shuffle(defect_data)
      defect_files[:], defect_filenames[:] = zip(*defect_data)

      conform_files = [(f, 0) for f in conform_files]
      defect_files = [(f, 1) for f in defect_files]

      # Calculate number of elements for each set based on the ratios
      total_conform = len(conform_files)
      train_conform_count = int(total_conform * train_params['ratio'])
      valid_conform_count = int(total_conform * valid_params['ratio'])

      total_defect = len(defect_files)
      train_defect_count = int(total_defect * train_params['ratio'])
      valid_defect_count = int(total_defect * valid_params['ratio'])

      # Split conform files and filenames into train, valid, and test sets
      train_conform_files = conform_files[:train_conform_count]
      train_conform_filenames = conform_filenames[:train_conform_count]

      valid_conform_files = conform_files[train_conform_count:train_conform_count + valid_conform_count]
      valid_conform_filenames = conform_filenames[train_conform_count:train_conform_count + valid_conform_count]

      test_conform_files = conform_files[train_conform_count + valid_conform_count:]
      test_conform_filenames = conform_filenames[train_conform_count + valid_conform_count:]

      # Split defect files and filenames into train, valid, and test sets
      train_defect_files = defect_files[:train_defect_count]
      train_defect_filenames = defect_filenames[:train_defect_count]

      valid_defect_files = defect_files[train_defect_count:train_defect_count + valid_defect_count]
      valid_defect_filenames = defect_filenames[train_defect_count:train_defect_count + valid_defect_count]

      test_defect_files = defect_files[train_defect_count + valid_defect_count:]
      test_defect_filenames = defect_filenames[train_defect_count + valid_defect_count:]

      # Sets creations
      test_files = test_conform_files + test_defect_files
      test_filenames = test_conform_filenames + test_defect_filenames

      if anomaly_detection:
          if anomaly_detection_couple:
              train_files = train_conform_files + train_defect_files
              train_filenames = train_conform_filenames + train_defect_filenames

              valid_files = valid_conform_files + valid_defect_files
              valid_filenames = valid_conform_filenames + valid_defect_filenames

              test_threshold_files = valid_conform_files + valid_defect_files
              test_threshold_filenames = valid_conform_filenames + valid_defect_filenames
          else:
              train_files = train_conform_files
              train_filenames = train_conform_filenames

              valid_files = valid_conform_files
              valid_filenames = valid_conform_filenames

              test_extension_files = train_defect_files
              test_extension_filenames = train_defect_filenames

              test_threshold_files = valid_conform_files + valid_defect_files
              test_threshold_filenames = valid_conform_filenames + valid_defect_filenames

      else:
          train_files = train_conform_files + train_defect_files
          train_filenames = train_conform_filenames + train_defect_filenames

          valid_files = valid_conform_files + valid_defect_files
          valid_filenames = valid_conform_filenames + valid_defect_filenames

      # Process each set
      train_images, train_names = train_files, train_filenames
      valid_images, valid_names = valid_files, valid_filenames
      test_images, test_names = test_files, test_filenames

      if anomaly_detection:
          if not anomaly_detection_couple:
              test_extension_images, test_extension_names = test_extension_files, test_extension_filenames
          test_threshold_images, test_threshold_names = test_threshold_files, test_threshold_filenames

      # Definire le coppie di liste da mescolare
      sets_to_shuffle = [
          (test_images, test_names),
          (train_images, train_names),
          (valid_images, valid_names)
      ]

      if anomaly_detection:
          if not anomaly_detection_couple:
              sets_to_shuffle.extend([
                  (test_extension_images, test_extension_names),
                  (test_threshold_images, test_threshold_names)
              ])
          sets_to_shuffle.extend([
              (test_threshold_images, test_threshold_names)
          ])

      # Mescolare direttamente le coppie di liste
      for images, names in sets_to_shuffle:
          combined = list(zip(images, names))
          random.shuffle(combined)
          images[:], names[:] = zip(*combined)

      if anomaly_detection and not anomaly_detection_couple:
          test_extended_images = test_images + test_extension_images
          test_extended_names = test_names + test_extension_names

      # Create datasets
      train_dataset = PanelsDataset(train_images)
      valid_dataset = PanelsDataset(valid_images)
      test_dataset = PanelsDataset(test_images)

      if anomaly_detection:
          if not anomaly_detection_couple:
              test_extended_dataset = PanelsDataset(test_extended_images)
          test_threshold_dataset = PanelsDataset(test_threshold_images)

      if anomaly_detection:
          if not anomaly_detection_couple:
              return (
                  (train_dataset, train_names),
                  (valid_dataset, valid_names),
                  (test_dataset, test_names),
                  (test_extended_dataset, test_extension_names),
                  (test_threshold_dataset, test_threshold_names)
              )
          return (
              (train_dataset, train_names),
              (valid_dataset, valid_names),
              (test_dataset, test_names),
              (test_threshold_dataset, test_threshold_names)
          )
      else:
          return (
              (train_dataset, train_names),
              (valid_dataset, valid_names),
              (test_dataset, test_names)
          )


      return None
    else:
      return None
      transform = transforms.ToTensor()  # Common transform for all images

      conform_path = os.path.join(dataset_path, 'normal')
      defect_path = os.path.join(dataset_path, 'defects')

      out_conform_path = os.path.join(dataset_path, "Cropped", 'normal')
      out_defect_path = os.path.join(dataset_path, "Cropped", 'defects')

      Path(out_conform_path).mkdir(parents=True, exist_ok=True)
      Path(out_defect_path).mkdir(parents=True, exist_ok=True)

      conform_filenames = sorted(os.listdir(conform_path))
      defect_filenames = sorted(os.listdir(defect_path))

      conform_files = [Image.open(os.path.join(conform_path, file)).convert('L') for file in conform_filenames]
      defect_files = [Image.open(os.path.join(defect_path, file)).convert('L') for file in defect_filenames]

      # Shuffle conform_files and conform_filenames while maintaining correspondence
      conform_data = list(zip(conform_files, conform_filenames))
      random.shuffle(conform_data)
      conform_files[:], conform_filenames[:] = zip(*conform_data)

      # Shuffle defect_files and defect_filenames while maintaining correspondence
      defect_data = list(zip(defect_files, defect_filenames))
      random.shuffle(defect_data)
      defect_files[:], defect_filenames[:] = zip(*defect_data)

      conform_images, conform_names = create_subimages_and_names(conform_files, conform_filenames, conform_files, subimage_width, train_params['subimage_shift'])
      defect_images, defect_names = create_subimages_and_names(defect_files, defect_filenames, conform_files, subimage_width, train_params['subimage_shift'])

      for idx in range(len(conform_images)):
        conform_images[idx][0].save(os.path.join(out_conform_path, f"cropped_{str(idx)}_"+conform_names[idx]))
      for idx in range(len(defect_images)):
        defect_images[idx][0].save(os.path.join(out_defect_path, f"cropped_{str(idx)}_"+defect_names[idx]))

      return None
session_info = {}

# User-specified parameters
dataset_path = base_path + 'Datasets/Dataset/product_a_32x128'

seed = 0
anomaly_detection = True         # Set to True or False based on your requirement
anomaly_detection_couple = True  # Set to True or False based on your requirement
subimage_width = 128             # Width for subimages or None

# Example parameters for each set
train_params = {'ratio': 0.7, 'subimage_shift': 50} # subimage_shift is irrelevant when not creating subimages
valid_params = {'ratio': 0.1, 'subimage_shift': 50} # subimage_shift is irrelevant when not creating subimages
test_params = {'ratio': 0.2, 'subimage_shift': 50} # subimage_shift is irrelevant when not creating subimages

# Create the datasets
datasets = prepare_datasets(
    dataset_path, train_params, valid_params, test_params,
    seed, anomaly_detection, anomaly_detection_couple, subimage_width, load_dataset=True
    )

train_dataset, train_names = datasets[0]
valid_dataset, valid_names = datasets[1]
test_dataset, test_names = datasets[2]
if anomaly_detection:
    if not anomaly_detection_couple:
        test_extended_dataset, test_extension_names = datasets[3]
        test_threshold_dataset, test_threshold_names = datasets[4]
    else:
        test_threshold_dataset, test_threshold_names = datasets[3]

session_info['random seed'] = seed
session_info['anomaly detection'] = anomaly_detection
session_info['subimage width'] = subimage_width
session_info['train_params'] = train_params
session_info['valid_params'] = valid_params
session_info['test_params'] = test_params

# Print the sizes of the datasets
print(f"Training set size: {len(train_dataset)}")
print(f"Validation set size: {len(valid_dataset)}")
print(f"Test set size: {len(test_dataset)}")
if anomaly_detection:
    print(f"Threshold set size: {len(test_threshold_dataset)}")
    if not anomaly_detection_couple:
        print(f"Test extended set size: {len(test_extended_dataset)}")

Unzipping <DATA_ROOT>/product_a_32x128/Zipped/normal
Unzipping <DATA_ROOT>/product_a_32x128/Zipped/defects
['cropped_corr_0_sub_0.png', 'cropped_corr_0_sub_1.png', 'cropped_corr_0_sub_10.png', 'cropped_corr_0_sub_11.png', 'cropped_corr_0_sub_12.png', 'cropped_corr_0_sub_13.png', 'cropped_corr_0_sub_14.png', 'cropped_corr_0_sub_15.png', 'cropped_corr_0_sub_16.png', 'cropped_corr_0_sub_17.png', 'cropped_corr_0_sub_18.png', 'cropped_corr_0_sub_19.png', 'cropped_corr_0_sub_2.png', 'cropped_corr_0_sub_20.png', 'cropped_corr_0_sub_3.png', 'cropped_corr_0_sub_4.png', 'cropped_corr_0_sub_5.png', 'cropped_corr_0_sub_6.png', 'cropped_corr_0_sub_7.png', 'cropped_corr_0_sub_8.png', 'cropped_corr_0_sub_9.png',  ... [20731 characters truncated]
Load image Ok
Training set size: 1758
Validation set size: 251
Test set size: 504
Threshold set size: 251


In [ ]:
import matplotlib.pyplot as plt
import random

def show_image(dataset, title):
    # Choose a random index
    chosen_index = 0 # random.randint(0, len(dataset) - 1)

    # Retrieve the image and its label
    image_tensor, label = dataset[chosen_index]
    # Convert the tensor to numpy array and adjust the dimensions
    image_np = image_tensor.numpy().transpose((1, 2, 0))

    # Plot the image
    plt.imshow(image_np)
    plt.title(f'{title} - Label: {"Defect" if label == 1 else "Normal"}')
    plt.show()

# Show a random image from each dataset
show_image(train_dataset, "Training Set")
show_image(valid_dataset, "Validation Set")
show_image(test_dataset, "Test Set")
if anomaly_detection:
    if not anomaly_detection_couple:
        show_image(test_extended_dataset, "Test Extended Set")
        show_image(test_threshold_dataset, "Test Threshold Set")
    else:
        show_image(test_threshold_dataset, "Test Threshold Set")

In [ ]:
from torch.utils.data import DataLoader

# Set the batch size
batch_size = 32

# Create DataLoaders for training, validation, and test sets
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
if anomaly_detection:
    if not anomaly_detection_couple:
        test_extended_loader = DataLoader(test_extended_dataset, batch_size=batch_size, shuffle=False)
        test_threshold_loader = DataLoader(test_threshold_dataset, batch_size=batch_size, shuffle=False)
    else:
        test_threshold_loader = DataLoader(test_threshold_dataset, batch_size=batch_size, shuffle=False)

session_info['batch size'] = batch_size

In [ ]:
idx = 11 # 8, 25
if anomaly_detection_couple:
    for images, labels in train_loader:
        image_couple = images[idx]
        label_couple = labels[idx]
        break

print(image_couple.shape)

image_np = image_couple.numpy().transpose((1, 2, 0))

# Plot the image
plt.imshow(image_np)
plt.title(f'Reference image - Label: {"Defect" if label_couple == 1 else "Normal"}')
plt.show()

torch.Size([1, 32, 128])


In [ ]:
import torch
import torch.nn as nn
from torch.nn import MSELoss

class AnomalyDetector(nn.Module): # Variant with Upsample + Conv2d instead of ConvTranspose2d
    K1 = 20
    K2 = 40
    def __init__(self, size=(32,128)):
        Ke = 3
        Kd = 4
        self.size = size
        # 3x3 kernel with 1x1 padding for the upsampling convolutions (mirrors the encoder).
        KernelUp = 3
        PadUp = 1

        super(AnomalyDetector, self).__init__()
        # Encoder (unchanged)
        self.encoder = nn.Sequential(
            nn.Conv2d(1, self.K1, (Ke, Ke), stride=(2, 2), padding=(1, 1)),
            nn.BatchNorm2d(self.K1),
            nn.ReLU(),
            nn.Conv2d(self.K1, self.K2, (Ke, Ke), stride=(2, 2), padding=(1, 1)),
            nn.BatchNorm2d(self.K2),
            nn.ReLU(),
        )

        # Decoder with Upsample + Conv2d (TFLite-friendly, no transposed convolutions)
        self.decoder = nn.Sequential(
            # Step 1: x2 bilinear upsampling followed by a 3x3 convolution.
            # Step 2: upsample directly to the exact input size, so the output matches the input even when the
            # intermediate sizes are odd (avoids the off-by-two mismatch of a second x2 upsample).
            # Upsample+Conv2d is used instead of ConvTranspose2d because transposed convolutions with output
            # padding are not supported by the ai-edge-torch -> TFLite INT8 converter.
            nn.Upsample(size=(self.size[0], self.size[1]), mode='bilinear', align_corners=True), # upsample to the exact target output size
            nn.Conv2d(self.K1, 1, (KernelUp, KernelUp), stride=(1, 1), padding=(PadUp, PadUp)),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.loss = MSELoss()

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        mse_loss = self.loss(decoded, x)
        return decoded, mse_loss

In [ ]:
from torchinfo import summary

model = AnomalyDetector()
summary(model, input_size=(1, 1, 32, 128))

Layer (type:depth-idx)                   Output Shape              Param #
AnomalyDetector                          [1, 1, 32, 128]           --
├─Sequential: 1-1                        [1, 40, 8, 32]            --
│    └─Conv2d: 2-1                       [1, 20, 16, 64]           200
│    └─BatchNorm2d: 2-2                  [1, 20, 16, 64]           40
│    └─ReLU: 2-3                         [1, 20, 16, 64]           --
│    └─Conv2d: 2-4                       [1, 40, 8, 32]            7,240
│    └─BatchNorm2d: 2-5                  [1, 40, 8, 32]            80
│    └─ReLU: 2-6                         [1, 40, 8, 32]            --
├─Sequential: 1-2                        [1, 1, 32, 128]           --
│    └─Upsample: 2-7                     [1, 40, 16, 64]           --
│    └─Conv2d: 2-8                       [1, 20, 16, 64]           7,220
│    └─BatchNorm2d: 2-9                  [1, 20, 16, 64]           40
│    └─ReLU: 2-10                        [1, 20, 16, 64]           --
│    └─U

In [ ]:
import torch
from torch.optim import Adam, SGD, Adagrad, RMSprop, AdamW
from torch.nn import MSELoss
import json

# Initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Device:', device)

kernel_size = 3
model = AnomalyDetector().to(device)

epochs = 3000
train_path = base_path + 'projects/output/product_b' + '_couple' + f'_{subimage_width}' + f'_{train_params["subimage_shift"]}' + f'_N1' + f'_k{kernel_size}' + f'_AdamW' # + f'_{}' # + '_test' #
overwrite = True

session_info['kernel_size'] = kernel_size
session_info['epochs'] = epochs
session_info['train_path'] = train_path

def create_input_couples(inputs, labels, image_couple):
    inputs_couples = inputs.clone()
    for i, label in enumerate(labels):
        if label == 1:
            inputs_couples[i] = image_couple
    return inputs_couples

def train_model(session_info, model, train_loader, valid_loader, patience, learning_rate=0.001, epochs=2000, train_path=''):
    criterion = MSELoss()

    # Adam
    # optimizer = Adam(model.parameters(), lr=learning_rate)

    # Stochastic Gradient Descent (SGD)
    # optimizer = SGD(model.parameters(), lr=learning_rate, momentum=0.9)

    # SGD with Nesterov Accelerated Gradient (NAG)
    # optimizer = SGD(model.parameters(), lr=learning_rate, momentum=0.9, nesterov=True)

    # Adagrad
    # optimizer = Adagrad(model.parameters(), lr=learning_rate)

    # RMSprop
    # optimizer = RMSprop(model.parameters(), lr=learning_rate)

    # AdamW
    optimizer = AdamW(model.parameters(), lr=learning_rate)

    session_info['optimizer'] = str(optimizer).replace("\n","").replace("    ",", "),

    best_loss = float('inf')
    epochs_no_improve = 0

    session_info['learning, loss, epoch'] = [[learning_rate, best_loss, 0]]

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for inputs, labels in train_loader:
            if anomaly_detection_couple:
                inputs_couples = create_input_couples(inputs, labels, image_couple)
                inputs_couples = inputs_couples.to(device)
            inputs = inputs.to(device)
            optimizer.zero_grad()
            outputs, _ = model(inputs)
            if anomaly_detection_couple:
                loss = criterion(outputs, inputs_couples)
            else:
                loss = criterion(outputs, inputs)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        valid_loss = 0
        model.eval()
        with torch.no_grad():
            for inputs, labels in valid_loader:
                if anomaly_detection_couple:
                    inputs_couples = create_input_couples(inputs, labels, image_couple)
                    inputs_couples = inputs_couples.to(device)
                inputs = inputs.to(device)
                outputs, _ = model(inputs)
                if anomaly_detection_couple:
                    loss = criterion(outputs, inputs_couples)
                else:
                    loss = criterion(outputs, inputs)
                valid_loss += loss.item()

        avg_valid_loss = valid_loss / len(valid_loader)
        print(f'\rEpoch {epoch+1}, Train Loss: {train_loss/len(train_loader)}, Valid Loss: {avg_valid_loss}.', end='')

        torch.save(model.state_dict(), train_path + '/last_model.pth')

        # Check for improvement
        if avg_valid_loss < best_loss:
            best_loss = avg_valid_loss
            epochs_no_improve = 0
            best_model = model.state_dict()
            session_info['best model epoch'] = epoch+1
            session_info['best loss'] = best_loss
        else:
            epochs_no_improve += 1
            if epochs_no_improve == patience:
                print(f' Reducing learning rate.')
                learning_rate /= 10
                session_info['learning, loss, epoch'] += [[learning_rate, avg_valid_loss, epoch+1]]
                if learning_rate < 1e-6:  # Set a lower limit for the learning rate
                    print(f'\nEarly stopping triggered after {epoch+1} epochs!')
                    session_info['last epoch'] = epoch+1
                    break
                for param_group in optimizer.param_groups:
                    param_group['lr'] = learning_rate
                epochs_no_improve = 0  # Reset the no improvement counter
            else:
                print(f' Patience: {epochs_no_improve}.', end='')

    # Saving the best model
    torch.save(best_model, train_path + '/best_model.pth')
    model.load_state_dict(best_model)

try:
    os.makedirs(train_path, exist_ok=overwrite)
    train_en = True
except:
    train_en = False

    # Load existing model
    try:
      model_state_dict = torch.load(train_path + '/best_model.pth', map_location=torch.device(device))
      model.load_state_dict(model_state_dict)
      print('Model loaded:', train_path + '/best_model.pth')
    except:
      model_state_dict = torch.load(train_path + '/last_model.pth', map_location=torch.device(device))
      model.load_state_dict(model_state_dict)
      print('Model loaded:', train_path + '/last_model.pth')

if train_en:
    patience = 10
    session_info['patience'] = patience
    train_model(session_info, model, train_loader, valid_loader, patience, epochs=epochs, train_path=train_path)
    with open(train_path + '/session_info.json', "w") as file:
        json.dump(session_info, file, indent=4)
    print('\n\n' + json.dumps(session_info, indent=4))

Device: cuda
Epoch 111, Train Loss: 0.0023936807685955006, Valid Loss: 0.0027772801695391536. Reducing learning rate.
Epoch 1158, Train Loss: 0.0017699981489303438, Valid Loss: 0.0021486707701114938.

In [ ]:
crop = {
    "left": 5 or None,
    "right": -5 or None,
    "up": None,
    "down": None
}

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, auc, confusion_matrix

def test_model_and_construct_roc_curve(model, loader, path, loader_name=None, crop=None, best_results=None):
    model.eval()
    test_scores = []
    true_labels = []

    if not crop:
      crop = {
          "left": None,
          "right": None,
          "up": None,
          "down": None
      }

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs,_ = model(inputs)
            inputs = inputs[:,:,crop['up']:crop['down'],crop['left']:crop['right']]
            outputs = outputs[:,:,crop['up']:crop['down'],crop['left']:crop['right']]
            loss = torch.mean((outputs - inputs) ** 2, dim=[1, 2, 3])
            test_scores.extend(loss.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    # Calculation of ROC curve and thresholds
    fpr, tpr, thresholds = roc_curve(true_labels, test_scores)
    roc_auc = auc(fpr, tpr)

    # Create a DataFrame
    df = pd.DataFrame({
        'Threshold': thresholds,
        'FPR': fpr,
        'TPR': tpr
    })

    df['SPR'] = 1 - df['FPR']

    n = 2

    print(df.iloc[::n])

    if best_results == None:
        # Calculate the accuracy for each threshold and find the best one
        best_results = {
            'accuracy': 0,
            'rate': 0,
            'threshold': 0,
            'fpr': 0,
            'tpr': 0,
            'predictions': []
        }

        for i, threshold in enumerate(thresholds):
            predictions = [1 if score > threshold else 0 for score in test_scores]
            accuracy = np.mean([pred == true for pred, true in zip(predictions, true_labels)])

            if (tpr[i] - fpr[i]) > best_results['rate']:
                best_results['accuracy'] = accuracy
                best_results['rate'] = (tpr[i] - fpr[i])
                best_results['threshold'] = threshold
                best_results['predictions'] = predictions
                best_results['fpr'] = fpr[i]
                best_results['tpr'] = tpr[i]

    else:
        # Find the index of the threshold closest to best_results['threshold']
        closest_idx = min(range(len(thresholds)), key=lambda i: abs(thresholds[i] - best_results['threshold']))

        # Use the index to access the corresponding values
        threshold = thresholds[closest_idx]
        predictions = [1 if score > threshold else 0 for score in test_scores]
        accuracy = np.mean([pred == true for pred, true in zip(predictions, true_labels)])

        # Update best_results with the values found
        best_results['accuracy'] = accuracy
        best_results['rate'] = (tpr[closest_idx] - fpr[closest_idx])
        best_results['threshold'] = threshold
        best_results['predictions'] = predictions
        best_results['fpr'] = fpr[closest_idx]
        best_results['tpr'] = tpr[closest_idx]

    #else:
    # ROC curve plot
    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve, area = %0.5f' % roc_auc)
    plt.scatter(best_results['fpr'], best_results['tpr'], color='red', label=f'Best accuracy: {best_results["accuracy"]:.5f}')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    title = 'ROC'
    if loader_name != None:
        title = f'{loader_name} {title}'
    plt.title(title)
    plt.legend(loc="lower right")
    # plt.savefig(path + f'/roc_{loader_name}.pdf', format='pdf')
    plt.show()

    # Calculating and printing the confusion matrix
    cm = confusion_matrix(true_labels, best_results['predictions'])
    print('Confusion Matrix:\n', cm)
    print('\nRows: True classes, Columns: Predicted classes')
    print('Row/Column 0: Negative Class (e.g., Normal)')
    print('Row/Column 1: Positive Class (e.g., Anomaly)')

    return test_scores, best_results

# Model testing and ROC curve construction
test_scores, best_results = test_model_and_construct_roc_curve(model, test_threshold_loader, train_path, 'test_threshold_loader', crop)
test_scores, best_results = test_model_and_construct_roc_curve(model, test_loader, train_path, 'test_loader th', crop, best_results)
# test_scores, best_results = test_model_and_construct_roc_curve(model, test_loader, train_path, 'test_loader', crop)
# test_scores, best_results = test_model_and_construct_roc_curve(model, test_extended_loader, train_path, 'test_extended_loader th', crop)
#_est_scores, _est_results = test_model_and_construct_roc_curve(model, test_extended_loader, 'test_extended_loader', crop)

    Threshold       FPR       TPR       SPR
0         inf  0.000000  0.000000  1.000000
2    0.017000  0.000000  0.355556  1.000000
4    0.012004  0.000000  0.572222  1.000000
6    0.006482  0.000000  0.977778  1.000000
8    0.005625  0.014085  0.988889  0.985915
10   0.004237  0.070423  0.994444  0.929577
12   0.003528  0.112676  1.000000  0.887324


Confusion Matrix:
 [[ 71   0]
 [  5 175]]

Rows: True classes, Columns: Predicted classes
Row/Column 0: Negative Class (e.g., Normal)
Row/Column 1: Positive Class (e.g., Anomaly)
    Threshold       FPR       TPR       SPR
0         inf  0.000000  0.000000  1.000000
2    0.019055  0.000000  0.265928  1.000000
4    0.013124  0.000000  0.523546  1.000000
6    0.011458  0.000000  0.628809  1.000000
8    0.011427  0.006993  0.634349  0.993007
10   0.007606  0.013986  0.878116  0.986014
12   0.007520  0.020979  0.880886  0.979021
14   0.007315  0.027972  0.889197  0.972028
16   0.006320  0.034965  0.927978  0.965035
18   0.006238  0.041958  0.930748  0.958042
20   0.005344  0.048951  0.975069  0.951049
22   0.005226  0.055944  0.977839  0.944056
24   0.004782  0.069930  0.986150  0.930070
26   0.004487  0.083916  0.991690  0.916084
28   0.000430  1.000000  1.000000  0.000000


Confusion Matrix:
 [[139   4]
 [ 27 334]]

Rows: True classes, Columns: Predicted classes
Row/Column 0: Negative Class (e.g., Normal)
Row/Column 1: Positive Class (e.g., Anomaly)


In [ ]:
import tensorflow as tf
import ai_edge_torch
import numpy as np

model = model.cpu().eval()

sample_args = (torch.randn(1, 1, 32, 128, device='cpu'),)

def representative_data_gen():
  for inputs, _  in train_loader:
    yield [inputs[0].cpu()]

tfl_converter_flags = {
    'optimizations': [tf.lite.Optimize.DEFAULT],
    'representative_dataset': representative_data_gen,
    'target_spec': tf.lite.TargetSpec(
        supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    ),
    'inference_input_type': tf.int8,
    'inference_output_type': tf.int8
}

tflite_model_quant = ai_edge_torch.convert(
    model.eval(), sample_args, _ai_edge_converter_flags=tfl_converter_flags
)
tflite_file = train_path + '/model_20-40_32x128pixels_int8.tflite'
print(tflite_file)
tflite_model_quant.export(tflite_file)

<DATA_ROOT>/projects/output/product_b_couple_128_50_N1_k3_AdamW/model_20-40_32x128pixels_int8.tflite


In [ ]:
# Optional: export the same model to ONNX

import torch.onnx

model.eval()
model.to(device)
# dummy_input = test_dataset[0][0]
dummy_input = torch.randn(1, 1, test_dataset[0][0].shape[1], int(test_dataset[0][0].shape[2])) #, dset_1.shape[3]))
print(dummy_input.shape)
dummy_input = dummy_input.to(device)
print("Shape:", dummy_input.shape)

torch.onnx.export(model, dummy_input, train_path + "/model_12-25_32x128pixels.onnx", verbose=True)

In [ ]:
import matplotlib.pyplot as plt
import random
import torch
from sklearn.metrics import roc_curve

def show_reconstructions(model, dataset, test_scores, best_results, crop=None, num_examples=None):
    if not crop:
        crop = {
            "left": None,
            "right": None,
            "up": None,
            "down": None
        }

    model.eval()
    with torch.no_grad():
        indices = list(range(len(dataset)))
        random.shuffle(indices)

        if num_examples is not None:
            if num_examples > len(test_scores):
                num_examples = len(test_scores)
            indices = indices[:num_examples]

        for index in indices:
            image_tensor, label = dataset[index]
            anomaly_score = test_scores[index]
            is_anomaly = anomaly_score > best_results['threshold']
            predicted_label = 'Anomaly' if is_anomaly else 'Normal'

            image_tensor = image_tensor.unsqueeze(0).to(device)
            reconstructed, _ = model(image_tensor)  # the model returns (reconstruction, mse_loss)
            reconstructed = reconstructed.squeeze(0).cpu()

            original_np = image_tensor.squeeze(0).cpu().numpy().transpose((1, 2, 0))
            reconstructed_np = reconstructed.numpy().transpose((1, 2, 0))

            original_np = original_np[crop['up']:crop['down'],crop['left']:crop['right'], :]
            reconstructed_np = reconstructed_np[crop['up']:crop['down'],crop['left']:crop['right'], :]

            difference_np = np.abs(original_np - reconstructed_np)

            title_color = 'green' if (label == 1 and is_anomaly) or (label == 0 and not is_anomaly) else 'red'
            # if

            plt.figure(figsize=(12, 4))

            plt.subplot(1, 3, 1)
            plt.imshow(original_np) #, cmap='gray'
            plt.title(f'Original - Label: {"Defect" if label == 1 else "Normal"}')
            plt.axis('off')

            plt.subplot(1, 3, 2)
            plt.imshow(reconstructed_np) #, cmap='gray'
            plt.title('Reconstructed')
            plt.axis('off')

            ax = plt.subplot(1, 3, 3)
            plt.imshow(difference_np, cmap='hot') #, cmap='gray'
            # ax.set_title(f'Difference - Predicted: {predicted_label}', backgroundcolor=title_color)
            ax.set_title(f'Predicted: {predicted_label}', backgroundcolor=title_color) # Difference: {np.sum(difference_np) / np.prod(difference_np.shape):.9f}\nAnomaly Score: {anomaly_score:.9f}\n
            plt.axis('off')

            # plt.savefig(path + f'/roc_{loader_name}.pdf', format='pdf')
            plt.show()

print('Best threshold:', best_results['threshold'])
show_reconstructions(model, test_dataset, test_scores, best_results, crop=crop) # , num_examples=
# show_reconstructions(model, test_dataset, test_scores, best_results, crop=crop, num_examples=30)